# 02 — Pretraining Analysis

Visualise contrastive training curves, learned embedding space (t-SNE), and
modality contribution weights after Phase 1 pretraining.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

CKPT = Path('../checkpoints/pretrain/best.pt')
DATA = Path('../data')

## 1. Load pretrained model and extract embeddings

In [ ]:
from models.full_model import MultimodalPainModel
from data.datasets import BioVidDataset, NeonatalDataset
from torch.utils.data import DataLoader

if CKPT.exists():
    model = MultimodalPainModel(video_pretrained=False)
    ckpt  = torch.load(CKPT, map_location='cpu')
    model.video_enc.load_state_dict(ckpt['video_enc_state'],  strict=False)
    model.audio_enc.load_state_dict(ckpt['audio_enc_state'],  strict=False)
    model.physio_enc.load_state_dict(ckpt['physio_enc_state'], strict=False)
    model.eval()
    print(f'Loaded checkpoint: {CKPT}')
    print(f'Pretrain loss: {ckpt["loss"]:.4f}')
else:
    print(f'Checkpoint not found at {CKPT}. Run scripts/pretrain.py first.')
    model = None

## 2. t-SNE of fused embeddings (adult vs neonatal)

In [ ]:
if model is None:
    print('Skip — no model loaded.')
else:
    try:
        adult_ds = BioVidDataset(str(DATA / 'biovid'), binary=True)
        neo_ds   = NeonatalDataset(str(DATA / 'neonatal'))

        adult_loader = DataLoader(adult_ds, batch_size=16, shuffle=True)
        neo_loader   = DataLoader(neo_ds,   batch_size=16, shuffle=True)

        adult_embs, neo_embs = [], []
        adult_labels, neo_labels = [], []

        with torch.no_grad():
            for i, batch in enumerate(adult_loader):
                if i >= 10: break
                z = model.get_embeddings(batch['video'], batch['audio'], batch['physio'])
                adult_embs.append(z.numpy())
                adult_labels.extend(batch['label'].tolist())

            for i, batch in enumerate(neo_loader):
                if i >= 10: break
                z = model.get_embeddings(batch['video'], batch['audio'], batch['physio'])
                neo_embs.append(z.numpy())
                neo_labels.extend(batch['label'].tolist())

        adult_embs = np.concatenate(adult_embs)
        neo_embs   = np.concatenate(neo_embs)

        all_embs   = np.concatenate([adult_embs, neo_embs])
        domains    = ['Adult'] * len(adult_embs) + ['Neonatal'] * len(neo_embs)

        tsne = TSNE(n_components=2, perplexity=30, random_state=42)
        proj = tsne.fit_transform(all_embs)

        fig, ax = plt.subplots(figsize=(8, 6))
        colors = {'Adult': '#1E88E5', 'Neonatal': '#E53935'}
        for domain, color in colors.items():
            mask = np.array(domains) == domain
            ax.scatter(proj[mask, 0], proj[mask, 1], c=color, label=domain,
                       alpha=0.6, s=20, edgecolors='none')
        ax.set_title('t-SNE: Adult vs Neonatal Embeddings (Pre-Adaptation)')
        ax.legend()
        ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
        plt.tight_layout()
        plt.savefig('tsne_domain_gap.png', dpi=150)
        plt.show()
    except FileNotFoundError as e:
        print(f'Dataset not found: {e}')

## 3. Hyperparameter sensitivity — Temperature τ

In [ ]:
# Reproduce Figure 8a from the paper
tau_values  = [0.05, 0.10, 0.20, 0.50]
tau_acc     = [87.2, 89.4, 87.9, 84.3]   # from Table in Section 4.5
tau_std     = [2.8,  2.5,  2.7,  3.1]

fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(tau_values, tau_acc, yerr=tau_std, fmt='o-',
            color='#1E88E5', capsize=4, lw=2, ms=7)
ax.axvline(0.10, color='red', ls='--', alpha=0.7, label='Optimal τ=0.10')
ax.set_xlabel('Temperature τ')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Hyperparameter Sensitivity — Temperature (Adult Domain, 10-shot)')
ax.legend()
ax.set_ylim(80, 93)
plt.tight_layout()
plt.savefig('hyperparameter_temperature.png', dpi=150)
plt.show()